In [121]:
import torch
from torch.utils.data import DataLoader, random_split, TensorDataset
import logging
from torch import nn 

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [122]:
import urllib.request

# Download Tiny Shakespeare directly into your workspace
# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
file_path = "tinyshakespeare.txt"

# urllib.request.urlretrieve(url, file_path)

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length: {len(text)} characters")
print("Sample text:\n", text[:150])

Dataset length: 1115394 characters
Sample text:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A


In [123]:
itos, stoi = dict(), dict() 

words = text.split(' ')
unique_words = sorted(list(set(text.split(' '))))
vocab_size = len(unique_words)

# stoi and itos mappings 
stoi = {w: i for (i, w) in enumerate(unique_words)}
itos = {i: w for (i, w) in enumerate(unique_words)}

def tensor_to_sentence(token_tensor):
    if not isinstance(token_tensor, torch.Tensor):
        token_tensor = torch.tensor(token_tensor, dtype=torch.long)

    token_tensor = token_tensor.detach().cpu()

    if token_tensor.dim() == 0:
        return itos.get(int(token_tensor.item()), "<UNK>")

    if token_tensor.dim() == 1:
        return " ".join(itos.get(int(idx), "<UNK>") for idx in token_tensor.tolist())

    if token_tensor.dim() == 2:
        return [
            " ".join(itos.get(int(idx), "<UNK>") for idx in row.tolist())
            for row in token_tensor
        ]

    raise ValueError("Expected a 0D, 1D, or 2D tensor of token ids.")


data = torch.tensor([stoi[w] for w in words], dtype=torch.long)

# We need to create "chunks of text" 
seq_len = 64
n = data.shape[0]
X_list, Y_list = list(), list()
for i in range(0, len(data) - seq_len, seq_len):
    x = data[i: i + seq_len]
    y = data[i + 1: i + seq_len + 1]

    X_list.append(x)
    Y_list.append(y)


X = torch.stack(X_list, dim=0)
Y = torch.stack(Y_list, dim=0)


batch_size = 8
dataset = TensorDataset(X, Y)
train_size = int(.80*len(dataset))
test_size = int(.10*len(dataset))
val_size = len(dataset) - train_size - test_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, test_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  drop_last=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,  drop_last=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,  drop_last=True, pin_memory=True)



print(f"X tensor shape {X.shape}")
print(f"Y tensor shape {Y.shape}")
print(f"vocab size:  {vocab_size}")
assert torch.equal(X[:, 1:seq_len], Y[:, :seq_len - 1]), "Label != Next Token"

print(f"len(train_loader): {len(train_loader)}")
print(f"len(val_loader): {len(val_loader)}")
print(f"len(test_loader): {len(test_loader)}")



X tensor shape torch.Size([2654, 64])
Y tensor shape torch.Size([2654, 64])
vocab size:  42197
len(train_loader): 265
len(val_loader): 33
len(test_loader): 33


In [124]:
class RMSNorm(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        self.epsilon = 1e-6
        self.gamma = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor):
        mean = torch.mean(torch.pow(x, 2),dim=-1, keepdim=True)
        rms = torch.sqrt(mean + self.epsilon)
        rms_norm = (x / rms) * self.gamma
        return rms_norm


norm = RMSNorm(2)
norm.epsilon = 0


x = torch.tensor(
    [
        [[1, 1], [2, 2], [3, 3], [4, 4]],
        [[5, 5], [6, 6], [7, 7], [8, 8]]
    ], dtype=torch.float
)

expected = torch.tensor(
    [
        [[1, 1], [1, 1], [1, 1], [1, 1]],
        [[1, 1], [1, 1], [1, 1], [1, 1]],
    ], dtype=torch.float
)

o = norm(x)
assert torch.equal(o, expected), f"{o} != {expected}"


In [125]:
z = torch.tensor([[0, 0], [0, 0]])
o = torch.tensor([[1, 1,], [1, 1]])

s = torch.stack([z, o], dim=0)
print(s)
print(s.shape)


tensor([[[0, 0],
         [0, 0]],

        [[1, 1],
         [1, 1]]])
torch.Size([2, 2, 2])


In [126]:

x = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]], dtype=torch.float)

seq_len, d_model = x.shape[0], x.shape[1]


# Notes: 
# m - This value controls how much we rotate the embedding and it is depedent on the token's
#     location in the sequence. Token's later in the sequency are rotated furhter. 
# theta - This value controls the frequency at which we rotate. This value is predetermined, 
#         and is depedent only on d_model and the RoPE base frequency (10k). Theta is calculated 
#         for d_model/2 pairs and pairs that are near the beginning of the embedding rotate 
#         have 


i = torch.arange(0, d_model, 2, dtype=torch.float)
theta = torch.pow(10000, -i/d_model)
m = torch.arange(0, seq_len, dtype=torch.float)
rope_angles = m.unsqueeze(-1) * theta
print(f"rope_angles={rope_angles.shape}")


cos_raw = torch.cos(rope_angles)
sin_raw = torch.sin(rope_angles)
print(f"cos_raw={cos_raw.shape}")
print(f"sin_raw={sin_raw.shape}")

cos_expaned = torch.repeat_interleave(cos_raw, dim=1)
sin_expaned = torch.repeat_interleave(sin_raw, dim=1)
cos_expaned = torch.repeat_interleave(cos_raw, repeats=2, dim=1)
sin_expaned = torch.repeat_interleave(sin_raw, repeats=2, dim=1)

print(f"cos_expaned={cos_expaned.shape}")
print(f"sin_expaned={sin_expaned.shape}")


x_rotated = torch.rotate_half(x)

# R = torch.stack(
#     [
#         torch.stack([
#             torch.cos(rope_angles), -torch.sin(rope_angles)
#         ], dim=-1),
#         torch.stack([
#             torch.sin(rope_angles), torch.cos(rope_angles)
#         ], dim=-1)
#     ], dim=-1
# )


# print(rope_angles.shape)

# # print(R)
# print(R.shape)


# # cos = torch.cos(rope_angles)
# # sin = torch.sin(rope_angles)
# x_grouped = x.view(4, 2, 2).unsqueeze(-1)



# # print(R.shape)
# # print(x_grouped.shape)
# x_rotated = (R @ x_grouped)

# print(x_rotated.shape)

# print(x_rotated)

# print(x_rotated.shape)

# We have input of x of size 4 tokens with dim = 4

# We need to rotate each token's embedding 










rope_angles=torch.Size([3, 2])
cos_raw=torch.Size([3, 2])
sin_raw=torch.Size([3, 2])


TypeError: repeat_interleave() received an invalid combination of arguments - got (Tensor, dim=int), but expected one of:
 * (Tensor input, Tensor repeats, int dim = None, *, int output_size = None)
 * (Tensor repeats, *, int output_size = None)
 * (Tensor input, int repeats, int dim = None, *, int output_size = None)


In [ ]:


seq_len = 8 
qk = torch.rand(seq_len, seq_len)

q_masked = qk.masked_fill(mask, -torch.inf)
o = torch.softmax(q_masked, dim=-1)
print(o)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3990, 0.6010, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2283, 0.2554, 0.5163, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3186, 0.1837, 0.2776, 0.2200, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1181, 0.1305, 0.1817, 0.2887, 0.2809, 0.0000, 0.0000, 0.0000],
        [0.1932, 0.2235, 0.1414, 0.2267, 0.0973, 0.1180, 0.0000, 0.0000],
        [0.2296, 0.1126, 0.0960, 0.1344, 0.1291, 0.0870, 0.2112, 0.0000],
        [0.0839, 0.1041, 0.1722, 0.0849, 0.1363, 0.1140, 0.1650, 0.1396]])


In [ ]:
import math 
class AttentionHead(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # in and out set o d_model since this is a single head. 
        self.Q = torch.nn.Linear(d_model, d_model)
        self.K = torch.nn.Linear(d_model, d_model)
        self.V = torch.nn.Linear(d_model, d_model)

    def forward(self, x):
        q = self.Q(x)
        k = self.K(x)
        v = self.V(x)

        seq_len = x.shape[-2] #... x seq_len x dim_model
        mask = torch.triu(
            torch.ones((seq_len, seq_len), dtype=torch.bool),
            diagonal=1
        )

        qk = q @ k.transpose(-2, -1)
        qk_masked = qk.masked_fill(mask, -torch.inf)
        print(qk_masked)
        o = torch.softmax(qk_masked / math.sqrt(self.d_model), dim=-1) @ v
        return o







batch_size = 1
seq_len = 4 
d_model = 8 
x = torch.rand(batch_size, seq_len, d_model)

attention_head = AttentionHead(d_model)
o = attention_head(x)

print(o)





tensor([[[ 0.2701,    -inf,    -inf,    -inf],
         [ 0.0204,  0.0631,    -inf,    -inf],
         [-0.0452, -0.0214,  0.0119,    -inf],
         [ 0.2179,  0.1839,  0.2129,  0.2091]]], grad_fn=<MaskedFillBackward0>)
tensor([[[-0.0064, -0.6608, -0.2436, -0.2205,  0.3777,  0.7962, -1.1152,
           0.9523],
         [-0.1697, -0.5870, -0.1489, -0.2937,  0.1675,  0.6898, -0.9298,
           0.7369],
         [-0.1610, -0.6203, -0.0371, -0.2910,  0.2174,  0.5893, -0.8323,
           0.7008],
         [-0.1933, -0.5911, -0.0559, -0.3062,  0.1549,  0.5456, -0.6731,
           0.5866]]], grad_fn=<UnsafeViewBackward0>)


In [ ]:
from torch import optim

d_model = 8

# Create embedding. 

class FFN(nn.Module):

    def __init__(self, seq_len, d_model):
        super(FFN, self).__init__()
        self.seq = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        return self.seq(x)


class BabyTransformer(nn.Module):
    '''
    Single attention head transformer. 
    '''

    def __init__(self, seq_len, vocab_size, d_model):
        super(BabyTransformer, self).__init__()

        self.prenorm_1 = RMSNorm(d_model)
        self.head = AttentionHead(d_model)
        self.prenorm_2 = RMSNorm(d_model)
        self.ffn = FFN(seq_len, d_model)
        self.prenorm_3 = RMSNorm(d_model)
        self.linear = torch.nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x_norm = self.prenorm_1(x)
        head_o = self.head(x_norm) + x
        head_o_norm = self.prenorm_2(head_o)
        ffn_o = self.ffn(head_o_norm) + head_o
        ffn_o_norm = self.prenorm_3(ffn_o)
        logits = self.linear(ffn_o_norm) + ffn_o
        return logits 



model = BabyTransformer(seq_len, vocab_size, d_model)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=3,
)
epochs = 1
embedding = nn.Embedding(vocab_size, d_model)
for epoch in range(1):
        
    for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
        optimizer.zero_grad()

        emb = embedding(batch_x)
        logits = model(emb)


        print(f"{logits.shape=}")

        break 

        



    


tensor([[[ 0.3931,    -inf,    -inf,  ...,    -inf,    -inf,    -inf],
         [-0.4833, -1.4116,    -inf,  ...,    -inf,    -inf,    -inf],
         [ 0.8276,  1.4055,  1.3326,  ...,    -inf,    -inf,    -inf],
         ...,
         [ 0.3931,  0.0338, -0.9899,  ...,  0.3931,    -inf,    -inf],
         [ 1.5593,  0.5501, -1.6241,  ...,  1.5593,  0.4101,    -inf],
         [ 0.2310, -0.0679,  0.1575,  ...,  0.2310, -0.7521,  1.9776]],

        [[ 0.0309,    -inf,    -inf,  ...,    -inf,    -inf,    -inf],
         [-0.8800, -0.6342,    -inf,  ...,    -inf,    -inf,    -inf],
         [-0.3859,  1.5414,  0.6230,  ...,    -inf,    -inf,    -inf],
         ...,
         [ 0.5857,  0.0401, -0.3388,  ...,  0.3502,    -inf,    -inf],
         [ 1.1390,  0.9231,  1.3688,  ...,  0.7261,  1.9179,    -inf],
         [-0.6454, -0.1102,  0.4405,  ..., -1.1084,  0.6487,  0.3931]],

        [[ 0.2870,    -inf,    -inf,  ...,    -inf,    -inf,    -inf],
         [ 0.6884,  0.8058,    -inf,  ...,   

NotImplementedError: Module [ModuleList] is missing the required "forward" function